# Lab 6: NLP Fundamentals: Tokenization, Embeddings & Text Chunking
Mustaeen Ul Rehman | 6 November 2026

In [ ]:
!pip install -q scikit-learn sentence-transformers
import csv
import json
import math
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

## Task 1: Tokenization and a FAQ Chatbot

In [ ]:
samples = ["AI, search!", "I can't log in.", "help@example.com", "CS4944 version 2.1", "", "یہ ایک مثال ہے"]
for text in samples:
    whitespace = text.split()
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    print(repr(text))
    print("whitespace:", whitespace, len(whitespace))
    print("word-like:", words, len(words))

In [ ]:
t = time.perf_counter()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
model_load_seconds = time.perf_counter() - t
print("model load seconds:", round(model_load_seconds, 2))
print("max_seq_length:", model.max_seq_length)
for text in samples[:2]:
    print(repr(text), model.tokenizer.tokenize(text))

Two differences: split() keeps attached punctuation, so "late!" stays one token, while the word regex drops punctuation and splits contractions ("can't" becomes "can" and "t"). The regex also splits an email into parts (help, example, com) while split() keeps it whole. Normalization risk: lowercasing or stripping punctuation can damage case-sensitive identifiers such as CS4944 or API tokens, and splitting a contraction changes the shape of a negation, so the original text is kept for reporting.

In [ ]:
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records) if all(r.get(key) == value for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]

    def show_hits(self, hits):
        if not hits:
            print("No matching records.")
        for record, score in hits:
            print(record["id"], f"score={score:.3f}")
            print(record["text"])

In [ ]:
FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?", "answer": "Use the recovery link on the login page.", "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?", "answer": "Use the assignment upload page in the learning portal.", "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?", "answer": "Request an extension with medical evidence before the deadline.", "source": "handbook.txt"},
    {"id": "faq:portal", "question": "How do I access the learning portal?", "answer": "Open the portal link and sign in with your student account.", "source": "it_help.txt"},
    {"id": "faq:deadline", "question": "What is the assignment deadline policy?", "answer": "Late work is penalized unless an extension was approved.", "source": "handbook.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)

In [ ]:
engine = build_bot()
print(reply(engine, "How do I reset my password?"))
print(reply(engine, "password recovery"))
print(reply(engine, ""))
print(reply(engine, "volcano geology"))

The chatbot asks for confirmation on nonexact questions because a similarity score is not proof that an answer is correct. It shows candidates and lets the user pick the exact question, so a paraphrase or a close but wrong match is never presented as a verified answer.

## Task 2: Chunking and Semantic Suggestions

In [ ]:
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    cuts = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out


def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = sentence_chunks(text) if strategy == "sentence" else windows(text, 40, int(strategy))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id, start=a, end=b, text=text[a:b]))
    return result

In [ ]:
docs = {
    "late": "Late submission is normally penalized. Illness extensions require medical evidence before the deadline. Students must submit the medical certificate to the department office within three working days of the missed deadline, and the extension request must be filed before the original deadline passes whenever possible. A late request without medical evidence is treated as an ordinary late submission and loses marks under the normal penalty rule. The penalty is five marks per day for the first three days, and work submitted more than seven days late receives no marks at all unless an extension was approved in advance. Students who are ill during an examination week must contact the examinations office on the first day of illness and provide a doctor's note within five working days. The department keeps a record of every approved extension, including the reason, the new deadline and the name of the approving officer. Extensions are granted only for the period covered by the medical evidence, and no extension is granted for more than fourteen days in one semester without a faculty review. When a student is ill for more than one week during the teaching term, the department asks for a medical certificate that covers the full period of illness, a short written statement from the student explaining which assessments were missed, and the name of the teacher for each affected course, so that a single extension decision can be recorded for all of them.",
    "password": 'Reset a forgotten password using the recovery link. Never share recovery codes. The recovery link is sent to the registered email address and stays valid for thirty minutes, so students should request a new link if the old one has expired. After opening the link, choose a password with at least eight characters, including one number and one capital letter, and do not reuse a password from another website. The recovery codes given during enrolment are a backup for students who cannot access their email, and each code can be used only once. If all recovery codes are used, the student must visit the IT help desk with a student card to prove identity before a new set of codes is issued. The help desk never asks for a password by email or by phone, and any message that asks for a password should be reported as a phishing attempt. Students should change their password immediately if they suspect that someone else has used their account, and they should check the recent login activity on the portal. The portal locks the account for fifteen minutes after five failed login attempts, and the lock is removed automatically after that time.',
    "printer": 'For blank printer pages, check ink levels and run the cleaning utility. Print a test page afterwards. If the test page is also blank, check that the ink cartridges are installed correctly and that the protective tape has been removed from a new cartridge. The printer must be switched on and connected to the same network as the computer, and the correct printer must be selected in the print menu. A paper jam can also cause blank pages, so open the rear cover and remove any stuck paper gently without tearing it. If the problem continues after cleaning, run the print head alignment from the printer settings and print a second test page. The cleaning utility uses ink, so it should be run at most three times in one day. For a blank scanner image, check document placement and select the correct scanning source, because the scanner and the printer use different parts of the machine. If none of these steps help, note the printer model and the error light pattern and contact the IT help desk. Do not open the printer casing, because that can damage the device and end the warranty.',
}
evidence = {
    "late": "Illness extensions require medical evidence before the deadline.",
    "password": "Reset a forgotten password using the recovery link.",
    "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
}
for doc_id, text in docs.items():
    print(doc_id, "words:", len(text.split()))
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)
    total_words = sum(len(c["text"].split()) for c in chunks)
    original_words = sum(len(t.split()) for t in docs.values())
    sizes = [len(c["text"].split()) for c in chunks]
    print(strategy, "chunks:", len(chunks), "size min/max:", min(sizes), max(sizes), "repeated words:", total_words - original_words)
    for doc_id, ev in evidence.items():
        kept = any(ev in c["text"] for c in chunks if c["doc"] == doc_id)
        print(" ", doc_id, "evidence in one chunk:", kept)

In [ ]:
print(windows(""))
print(windows("short text"))
try:
    windows("a b", 2, 2)
except ValueError:
    print("invalid overlap rejected")
long_text = " ".join(["word"] * 95) + "."
print([len(long_text[a:b].split()) for a, b in sentence_chunks(long_text)])

Sentence splitting fails on abbreviations such as "Dr." or "e.g.", where a full stop does not end a sentence, and on decimal numbers such as "version 2.1". The splitter used here cuts after any full stop followed by whitespace, so it can split one sentence into two chunks and break the evidence.

In [ ]:
records = make_chunks(docs, "sentence")
queries = ["How can I regain access to my account?", "What happens if I submit my work late because I was sick?", "My printer prints empty pages, what should I check?"]
for backend in ["tfidf", "embedding"]:
    engine = Retriever(records, backend)
    print("Backend:", backend)
    for q in queries:
        print("Query:", q)
        engine.show_hits(engine.search(q, k=2))

## Task 3: Evaluate a Document Retrieval Chatbot

In [ ]:
docs = {
    "late": "Late submission is normally penalized. Illness extensions require medical evidence before the deadline. Students must submit the medical certificate to the department office within three working days of the missed deadline, and the extension request must be filed before the original deadline passes whenever possible. A late request without medical evidence is treated as an ordinary late submission and loses marks under the normal penalty rule. The penalty is five marks per day for the first three days, and work submitted more than seven days late receives no marks at all unless an extension was approved in advance. Students who are ill during an examination week must contact the examinations office on the first day of illness and provide a doctor's note within five working days. The department keeps a record of every approved extension, including the reason, the new deadline and the name of the approving officer. Extensions are granted only for the period covered by the medical evidence, and no extension is granted for more than fourteen days in one semester without a faculty review. When a student is ill for more than one week during the teaching term, the department asks for a medical certificate that covers the full period of illness, a short written statement from the student explaining which assessments were missed, and the name of the teacher for each affected course, so that a single extension decision can be recorded for all of them.",
    "password": 'Reset a forgotten password using the recovery link. Never share recovery codes. The recovery link is sent to the registered email address and stays valid for thirty minutes, so students should request a new link if the old one has expired. After opening the link, choose a password with at least eight characters, including one number and one capital letter, and do not reuse a password from another website. The recovery codes given during enrolment are a backup for students who cannot access their email, and each code can be used only once. If all recovery codes are used, the student must visit the IT help desk with a student card to prove identity before a new set of codes is issued. The help desk never asks for a password by email or by phone, and any message that asks for a password should be reported as a phishing attempt. Students should change their password immediately if they suspect that someone else has used their account, and they should check the recent login activity on the portal. The portal locks the account for fifteen minutes after five failed login attempts, and the lock is removed automatically after that time.',
    "assignment": "Upload assignments through the learning portal. Submit the final file as a PDF. The portal accepts files up to fifty megabytes, and the file name should include the student roll number and the assignment number so that the teacher can identify the work. Students must check the submission receipt on the portal after uploading, because an upload without a receipt is not counted as submitted. The deadline shown on the portal is the final deadline, and the portal stops accepting files at that exact time. Drafts can be saved and replaced before the deadline, but only the last uploaded file is marked. The assignment must be written in the student's own words, and any copied text must be quoted with a reference. Group assignments need the names and roll numbers of all members on the first page, and only one member should upload the file for the whole group. If the portal is down near the deadline, students should email the assignment to the teacher before the deadline and upload it as soon as the portal works again. Late work is penalized under the normal late submission rule unless an extension was approved.",
    "contractor": 'Contractors can work remotely only if the signed contract permits it. Employee rules do not apply to contractors. A contractor who works remotely without a written permission in the contract may lose the payment for those days, and the company is not required to provide office equipment for home use. The contract must state the number of remote days allowed each week, the working hours, and the method for reporting completed work. Contractors must attend the office for meetings when the manager asks, even on a remote day, unless the contract says otherwise. Remote work does not change the payment terms or the length of the contract, and the contractor is responsible for a stable internet connection and a safe workplace at home. Any dispute about remote work is decided from the signed contract only, and verbal promises are not accepted as evidence. The human resources office keeps a copy of every contractor agreement, and the contractor should keep a signed copy as well. Contractors who need to change their remote days must request the change in writing at least one week before the new arrangement starts.',
    "employee": 'Employees may request remote work with manager approval. Remote work is allowed for two days each week, and the employee must be available online during the normal working hours. The manager can refuse a remote request when the team needs the employee in the office for a project deadline or for training new staff. Employees who work remotely must attend the weekly team meeting in person unless the manager agrees otherwise. The company provides a laptop and a secure connection for remote work, and the employee must report any technical problem to the IT help desk on the same day. Remote work does not reduce the workload or change the leave rules, and the employee must record working hours in the same system used in the office. An employee who works remotely for more than four weeks in a row must meet the manager to review the arrangement. These rules apply to full-time employees only, and they do not cover contractors, whose remote work is decided by the signed contract. The human resources office updates this policy once a year, and the latest version is always the one on the staff portal.',
    "printer": 'For blank printer pages, check ink levels and run the cleaning utility. Print a test page afterwards. If the test page is also blank, check that the ink cartridges are installed correctly and that the protective tape has been removed from a new cartridge. The printer must be switched on and connected to the same network as the computer, and the correct printer must be selected in the print menu. A paper jam can also cause blank pages, so open the rear cover and remove any stuck paper gently without tearing it. If the problem continues after cleaning, run the print head alignment from the printer settings and print a second test page. The cleaning utility uses ink, so it should be run at most three times in one day. For a blank scanner image, check document placement and select the correct scanning source, because the scanner and the printer use different parts of the machine. If none of these steps help, note the printer model and the error light pattern and contact the IT help desk. Do not open the printer casing, because that can damage the device and end the warranty.',
}
queries = [
    {"query": "What is required for illness extensions?", "doc": "late", "evidence": "Illness extensions require medical evidence before the deadline.", "split": "dev"},
    {"query": "How do I reset a forgotten password?", "doc": "password", "evidence": "Reset a forgotten password using the recovery link.", "split": "dev"},
    {"query": "Where do I upload assignments?", "doc": "assignment", "evidence": "Upload assignments through the learning portal.", "split": "dev"},
    {"query": "Can contractors work remotely?", "doc": "contractor", "evidence": "Contractors can work remotely only if the signed contract permits it.", "split": "dev"},
    {"query": "What do I do for blank printer pages?", "doc": "printer", "evidence": "For blank printer pages, check ink levels and run the cleaning utility.", "split": "test"},
    {"query": "How can I recover my login access?", "doc": "password", "evidence": "Reset a forgotten password using the recovery link.", "split": "test"},
    {"query": "How should I hand in my coursework?", "doc": "assignment", "evidence": "Upload assignments through the learning portal.", "split": "test"},
    {"query": "What proof is needed when I was ill?", "doc": "late", "evidence": "Illness extensions require medical evidence before the deadline.", "split": "test"},
    {"query": "What are tomorrow's flight prices?", "doc": None, "evidence": None, "split": "test"},
    {"query": "Who won the football match?", "doc": None, "evidence": None, "split": "test"},
]
for q in queries:
    assert q["split"] in {"dev", "test"}
    if q["doc"] is not None:
        assert q["evidence"] and q["evidence"] in docs[q["doc"]]
with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)
with open("queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)
print("documents:", len(docs))
print("queries:", len(queries))
for doc_id, text in docs.items():
    print(doc_id, "words:", len(text.split()))

In [ ]:
def evaluate(scores, chunks, items):
    hits = []
    reciprocal = []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [rank for rank, i in enumerate(top, 1) if chunks[i]["doc"] == q["doc"] and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)
    if not hits:
        return None, None
    return float(np.mean(hits)), float(np.mean(reciprocal))


query_texts = [q["query"] for q in queries]
dev_results = {}
all_scores = {}
logs = []
print("model load seconds:", round(model_load_seconds, 2))
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)
    texts = [c["text"] for c in chunks]
    lengths = [len(model.tokenizer(t, truncation=False)["input_ids"]) for t in texts + query_texts]
    print(strategy, "chunks:", len(chunks), "max tokens:", max(lengths), "limit:", model.max_seq_length)
    if max(lengths) > model.max_seq_length:
        raise ValueError("Model limit exceeded; rechunk or shorten query")
    for method in ["tfidf", "embedding"]:
        t = time.perf_counter()
        if method == "tfidf":
            vectorizer = TfidfVectorizer()
            C = vectorizer.fit_transform(texts)
            Q = vectorizer.transform(query_texts)
        else:
            C = model.encode(texts, normalize_embeddings=True)
            Q = model.encode(query_texts, normalize_embeddings=True)
        encode_seconds = time.perf_counter() - t
        t = time.perf_counter()
        scores = (Q @ C.T).toarray() if method == "tfidf" else Q @ C.T
        search_seconds = time.perf_counter() - t
        all_scores[(strategy, method)] = scores
        for split in ["dev", "test"]:
            indices = [i for i, q in enumerate(queries) if q["split"] == split]
            result = evaluate(scores[indices], chunks, [queries[i] for i in indices])
            print(strategy, method, split, "Hit/MRR:", result, "encode/search seconds:", round(encode_seconds, 3), round(search_seconds, 3))
            if split == "dev":
                dev_results[(strategy, method)] = result
        for row, q in zip(scores, queries):
            for rank, i in enumerate(np.argsort(-row, kind="stable")[:3], 1):
                c = chunks[i]
                logs.append(dict(strategy=strategy, method=method, query=q["query"], split=q["split"], rank=rank, score=float(row[i]), chunk_id=c["id"], doc=c["doc"], start=c["start"], end=c["end"], text=c["text"], answerable=q["doc"] is not None))

In [ ]:
with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(logs[0]))
    writer.writeheader()
    writer.writerows(logs)
print("log rows:", len(logs))
best = max(dev_results, key=lambda k: (dev_results[k][0], dev_results[k][1]))
best_strategy, best_method = best
print("selected:", best_strategy, best_method, "dev Hit/MRR:", dev_results[best])
chunks = make_chunks(docs, best_strategy)
scores = all_scores[best]
for qi, q in enumerate(queries):
    print("Query:", q["query"])
    for rank, i in enumerate(np.argsort(-scores[qi], kind="stable")[:3], 1):
        c = chunks[i]
        print(" ", rank, c["id"], round(float(scores[qi][i]), 3), c["text"][:80])

Strategy justification and failures are read from the printed results above: the selected configuration is the one with the best development Hit@3 and MRR@3, and its test results are reported without retuning. An unanswerable query still returns text under embeddings because ranking is relative: the top result is the closest chunk, not a verified answer. TF-IDF returns nothing only when no word overlaps. No universal cosine cutoff was used; a threshold would need calibration on separate development examples.

In [ ]:
def answer(engine, query):
    hits = engine.search(query)
    if not hits:
        print("No candidate evidence; rephrase or contact a human.")
        return
    print("Candidate evidence; relevance must be checked:")
    for number, (r, score) in enumerate(hits, 1):
        print(number, r["id"], f"{score:.3f}", "characters", r["start"], r["end"])
    r, score = hits[0]
    print("Retrieved passage:", r["text"])
    print("Source:", r["doc"], "characters", r["start"], r["end"])


engine = Retriever(make_chunks(docs, best_strategy), best_method)
for q in ["How do I reset a forgotten password?", "What are tomorrow's flight prices?", "Can contractors work remotely?"]:
    print("Question:", q)
    answer(engine, q)

FAQ matching answers from a fixed list of questions: an exact question returns a stored answer and anything else shows candidates for confirmation. Document retrieval ranks chunks from the whole corpus, so it can return a related but wrong passage, and conflicting sources (the contractor rule and the employee rule) can both appear. That is why the chatbot shows candidates with sources and offsets instead of asserting an answer.